# 🤖 Churn Prediction Agentic Pipeline
### Auto EDA → Auto Preprocessing → Model Selection → Segmentation → Business Recommendations
---
This notebook runs as an **autonomous agent** — each agent makes decisions on its own.
Just run all cells and the pipeline handles everything automatically.

## 📦 Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

print('✅ All libraries imported successfully!')

---
## 🔍 AGENT 1 — Auto EDA Agent
*Automatically explores the dataset and reports findings.*

In [ ]:
class AutoEDAAgent:
    """
    Agent 1: Automatically performs Exploratory Data Analysis.
    Detects shape, missing values, class imbalance, and key distributions.
    """

    def __init__(self, filepath):
        self.filepath = filepath
        self.df = None
        self.report = {}

    def run(self):
        print('=' * 60)
        print('🔍 AGENT 1: AUTO EDA AGENT STARTED')
        print('=' * 60)

        self._load_data()
        self._check_shape()
        self._check_missing()
        self._check_class_balance()
        self._plot_distributions()
        self._plot_correlation()

        print('\n✅ AGENT 1 COMPLETE — EDA done!')
        return self.df, self.report

    def _load_data(self):
        self.df = pd.read_excel(self.filepath)
        print(f'\n📂 Loaded: {self.filepath}')
        print(f'   Shape: {self.df.shape}')

    def _check_shape(self):
        self.report['rows'] = self.df.shape[0]
        self.report['cols'] = self.df.shape[1]
        print(f'\n📊 Dataset: {self.report["rows"]} rows × {self.report["cols"]} columns')

    def _check_missing(self):
        missing = self.df.isnull().sum()
        missing = missing[missing > 0]
        self.report['missing_cols'] = missing.to_dict()
        if len(missing) == 0:
            print('\n✅ No missing values found!')
        else:
            print(f'\n⚠️  Missing values detected:')
            print(missing)
            # Visualise
            plt.figure(figsize=(10, 4))
            missing.plot(kind='bar', color='salmon')
            plt.title('Missing Values per Column')
            plt.ylabel('Count')
            plt.tight_layout()
            plt.show()

    def _check_class_balance(self):
        counts = self.df['Churn Label'].value_counts()
        pct = self.df['Churn Label'].value_counts(normalize=True) * 100
        self.report['churn_balance'] = pct.to_dict()
        print(f'\n⚖️  Class Balance:')
        for label, p in pct.items():
            print(f'   {label}: {p:.1f}%')
        # Decision
        minority_pct = pct.min()
        if minority_pct < 30:
            self.report['imbalanced'] = True
            print('   🚨 DECISION: Data is IMBALANCED → will use class_weight=balanced in models')
        else:
            self.report['imbalanced'] = False
            print('   ✅ DECISION: Data is balanced')

        plt.figure(figsize=(5, 4))
        counts.plot(kind='bar', color=['steelblue', 'salmon'])
        plt.title('Churn Label Distribution')
        plt.ylabel('Count')
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

    def _plot_distributions(self):
        num_cols = ['Tenure Months', 'Monthly Charges']
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        for ax, col in zip(axes, num_cols):
            sns.histplot(self.df[col], kde=True, ax=ax, color='steelblue')
            ax.set_title(f'Distribution: {col}')
        plt.suptitle('Key Numeric Feature Distributions', fontsize=13)
        plt.tight_layout()
        plt.show()

    def _plot_correlation(self):
        df_temp = self.df.copy()
        df_temp['Total Charges'] = pd.to_numeric(df_temp['Total Charges'], errors='coerce')
        num_df = df_temp[['Tenure Months', 'Monthly Charges', 'Total Charges',
                           'Churn Value', 'Churn Score', 'CLTV']].dropna()
        plt.figure(figsize=(7, 5))
        sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
        plt.title('Correlation Heatmap')
        plt.tight_layout()
        plt.show()


# ▶ RUN AGENT 1
eda_agent = AutoEDAAgent('Telco_customer_churn.xlsx')
raw_df, eda_report = eda_agent.run()

---
## 🛠️ AGENT 2 — Auto Preprocessing Agent
*Automatically cleans, encodes, and prepares data based on EDA findings.*

In [ ]:
class AutoPreprocessingAgent:
    """
    Agent 2: Automatically preprocesses data.
    Drops useless columns, fixes dtypes, encodes categoricals.
    """

    DROP_COLS = [
        'CustomerID', 'Count', 'Country', 'State', 'City',
        'Zip Code', 'Lat Long', 'Latitude', 'Longitude',
        'Churn Score', 'CLTV', 'Churn Reason', 'Churn Label'
    ]

    def __init__(self, df, eda_report):
        self.df = df.copy()
        self.eda_report = eda_report

    def run(self):
        print('=' * 60)
        print('🛠️  AGENT 2: AUTO PREPROCESSING AGENT STARTED')
        print('=' * 60)

        self._drop_columns()
        self._fix_dtypes()
        self._encode_categoricals()
        self._split_features_target()

        print('\n✅ AGENT 2 COMPLETE — Data ready for modelling!')
        return self.X, self.Y, self.df

    def _drop_columns(self):
        existing = [c for c in self.DROP_COLS if c in self.df.columns]
        self.df.drop(columns=existing, inplace=True)
        print(f'\n🗑️  Dropped {len(existing)} non-predictive columns')

    def _fix_dtypes(self):
        self.df['Total Charges'] = pd.to_numeric(self.df['Total Charges'], errors='coerce')
        before = self.df['Total Charges'].isnull().sum()
        self.df['Total Charges'].fillna(self.df['Total Charges'].median(), inplace=True)
        print(f'\n🔧 Fixed "Total Charges" dtype → filled {before} NaN(s) with median')

    def _encode_categoricals(self):
        cat_cols = self.df.select_dtypes(include='object').columns.tolist()
        le = LabelEncoder()
        for col in cat_cols:
            self.df[col] = le.fit_transform(self.df[col])
        print(f'\n🔢 Label-encoded {len(cat_cols)} categorical columns: {cat_cols}')

    def _split_features_target(self):
        self.X = self.df.drop(columns=['Churn Value'])
        self.Y = self.df['Churn Value']
        print(f'\n📐 Features X: {self.X.shape}  |  Target Y: {self.Y.shape}')


# ▶ RUN AGENT 2
prep_agent = AutoPreprocessingAgent(raw_df, eda_report)
X, Y, clean_df = prep_agent.run()

---
## 🏆 AGENT 3 — Auto Model Selection Agent
*Trains multiple models, compares them, and automatically picks the best one.*

In [ ]:
class AutoModelSelectionAgent:
    """
    Agent 3: Trains Random Forest, Gradient Boosting, and Logistic Regression.
    Automatically selects the best model based on Recall + F1.
    Also does feature selection and plots evaluation curves.
    """

    def __init__(self, X, Y, imbalanced=True):
        self.X = X
        self.Y = Y
        self.cw = 'balanced' if imbalanced else None
        self.best_model = None
        self.best_model_name = ''
        self.results = []
        self.X_train = self.X_test = self.Y_train = self.Y_test = None

    def run(self):
        print('=' * 60)
        print('🏆 AGENT 3: AUTO MODEL SELECTION AGENT STARTED')
        print('=' * 60)

        self._split()
        self._train_all_models()
        self._select_best()
        self._evaluate_best()
        self._plot_confusion_matrix()
        self._plot_roc_curve()
        self._feature_importance()

        print('\n✅ AGENT 3 COMPLETE — Best model selected!')
        return self.best_model, self.X_train, self.X_test, self.Y_train, self.Y_test

    def _split(self):
        self.X_train, self.X_test, self.Y_train, self.Y_test = train_test_split(
            self.X, self.Y, test_size=0.2, random_state=42
        )
        print(f'\n✂️  Train: {self.X_train.shape[0]} samples | Test: {self.X_test.shape[0]} samples')

    def _train_all_models(self):
        models = {
            'Random Forest': RandomForestClassifier(
                n_estimators=300, max_depth=10,
                random_state=42, class_weight=self.cw
            ),
            'Gradient Boosting': GradientBoostingClassifier(
                n_estimators=200, max_depth=5, random_state=42
            ),
            'Logistic Regression': LogisticRegression(
                max_iter=1000, random_state=42, class_weight=self.cw
            )
        }

        print('\n🔄 Training models...')
        self.trained_models = {}
        for name, model in models.items():
            model.fit(self.X_train, self.Y_train)
            y_pred = model.predict(self.X_test)
            row = {
                'Model': name,
                'Accuracy':  round(accuracy_score(self.Y_test, y_pred), 4),
                'Recall':    round(recall_score(self.Y_test, y_pred), 4),
                'Precision': round(precision_score(self.Y_test, y_pred), 4),
                'F1 Score':  round(f1_score(self.Y_test, y_pred), 4),
                'ROC-AUC':   round(roc_auc_score(self.Y_test, model.predict_proba(self.X_test)[:, 1]), 4)
            }
            self.results.append(row)
            self.trained_models[name] = model
            print(f'   ✔ {name} done')

        result_df = pd.DataFrame(self.results).sort_values(
            by=['Recall', 'F1 Score'], ascending=False
        )
        print('\n📊 Model Comparison:')
        print(result_df.to_string(index=False))
        self.result_df = result_df

    def _select_best(self):
        # Agent decision: pick by Recall (most important for churn)
        best_row = self.result_df.iloc[0]
        self.best_model_name = best_row['Model']
        self.best_model = self.trained_models[self.best_model_name]
        print(f'\n🏆 AGENT DECISION: Best model → "{self.best_model_name}"')
        print(f'   Recall={best_row["Recall"]} | F1={best_row["F1 Score"]} | ROC-AUC={best_row["ROC-AUC"]}')

        # Bar chart comparison
        metrics = ['Accuracy', 'Recall', 'Precision', 'F1 Score', 'ROC-AUC']
        plot_df = self.result_df.set_index('Model')[metrics]
        plot_df.T.plot(kind='bar', figsize=(10, 5), colormap='Set2')
        plt.title('Model Comparison — All Metrics')
        plt.ylabel('Score')
        plt.xticks(rotation=0)
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.show()

    def _evaluate_best(self):
        y_pred = self.best_model.predict(self.X_test)
        print(f'\n📋 Classification Report — {self.best_model_name}:')
        print(classification_report(self.Y_test, y_pred))

    def _plot_confusion_matrix(self):
        y_pred = self.best_model.predict(self.X_test)
        cm = confusion_matrix(self.Y_test, y_pred)
        plt.figure(figsize=(5, 4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['No Churn', 'Churn'],
                    yticklabels=['No Churn', 'Churn'])
        plt.title(f'Confusion Matrix — {self.best_model_name}')
        plt.ylabel('Actual')
        plt.xlabel('Predicted')
        plt.tight_layout()
        plt.show()

    def _plot_roc_curve(self):
        y_prob = self.best_model.predict_proba(self.X_test)[:, 1]
        fpr, tpr, _ = roc_curve(self.Y_test, y_prob)
        auc = roc_auc_score(self.Y_test, y_prob)
        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, color='steelblue', label=f'AUC = {auc:.3f}')
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve — {self.best_model_name}')
        plt.legend()
        plt.tight_layout()
        plt.show()

    def _feature_importance(self):
        if not hasattr(self.best_model, 'feature_importances_'):
            print('\n(Feature importance not available for this model)')
            return
        imp = pd.Series(self.best_model.feature_importances_, index=self.X.columns)
        imp = imp.sort_values(ascending=False).head(15)
        plt.figure(figsize=(10, 5))
        imp.plot(kind='bar', color='steelblue')
        plt.title('Top 15 Feature Importances')
        plt.ylabel('Importance')
        plt.tight_layout()
        plt.show()


# ▶ RUN AGENT 3
model_agent = AutoModelSelectionAgent(X, Y, imbalanced=eda_report.get('imbalanced', True))
best_model, X_train, X_test, Y_train, Y_test = model_agent.run()

---
## 🗂️ AGENT 4 — Customer Segmentation Agent
*Automatically finds optimal clusters and profiles each segment.*

In [ ]:
class SegmentationAgent:
    """
    Agent 4: Builds segmentation data, finds optimal k via Elbow + Silhouette,
    assigns cluster names, and visualises segments.
    """

    CLUSTER_NAMES = {
        0: 'Budget Loyal Customers',
        1: 'High Risk New Customers',
        2: 'Loyal Premium Customers'
    }

    def __init__(self, raw_df, best_model, X):
        self.raw_df = raw_df.copy()
        self.best_model = best_model
        self.X = X
        self.seg_data = None
        self.optimal_k = 3

    def run(self):
        print('=' * 60)
        print('🗂️  AGENT 4: SEGMENTATION AGENT STARTED')
        print('=' * 60)

        self._build_seg_data()
        self._find_optimal_k()
        self._fit_kmeans()
        self._name_clusters()
        self._visualise()

        print('\n✅ AGENT 4 COMPLETE — Segments ready!')
        return self.seg_data

    def _build_seg_data(self):
        df_temp = self.raw_df.copy()
        df_temp['Total Charges'] = pd.to_numeric(df_temp['Total Charges'], errors='coerce')
        df_temp['Total Charges'].fillna(df_temp['Total Charges'].median(), inplace=True)

        y_prob = self.best_model.predict_proba(self.X)[:, 1]
        self.seg_data = pd.DataFrame({
            'Tenure Months':    df_temp['Tenure Months'].values,
            'Monthly Charges':  df_temp['Monthly Charges'].values,
            'Total Charges':    df_temp['Total Charges'].values,
            'Churn Probability': y_prob
        })
        print(f'\n📊 Segmentation data built: {self.seg_data.shape}')

    def _find_optimal_k(self):
        scaler = StandardScaler()
        self.scaled = scaler.fit_transform(self.seg_data)

        wcss, sil_scores = [], []
        k_range = range(2, 10)
        for k in k_range:
            km = KMeans(n_clusters=k, random_state=42)
            labels = km.fit_predict(self.scaled)
            wcss.append(km.inertia_)
            sil_scores.append(silhouette_score(self.scaled, labels))

        # Pick k with highest silhouette score
        self.optimal_k = list(k_range)[np.argmax(sil_scores)]
        print(f'\n🔢 AGENT DECISION: Optimal k = {self.optimal_k} (highest Silhouette Score = {max(sil_scores):.3f})')

        fig, axes = plt.subplots(1, 2, figsize=(13, 4))
        axes[0].plot(k_range, wcss, marker='o', color='steelblue')
        axes[0].set_title('Elbow Method (WCSS)')
        axes[0].set_xlabel('k')
        axes[0].set_ylabel('WCSS')
        axes[1].plot(k_range, sil_scores, marker='o', color='green')
        axes[1].axvline(self.optimal_k, color='red', linestyle='--',
                        label=f'Optimal k={self.optimal_k}')
        axes[1].set_title('Silhouette Score')
        axes[1].set_xlabel('k')
        axes[1].set_ylabel('Score')
        axes[1].legend()
        plt.suptitle('Finding Optimal Number of Clusters', fontsize=13)
        plt.tight_layout()
        plt.show()

    def _fit_kmeans(self):
        km = KMeans(n_clusters=self.optimal_k, random_state=42)
        self.seg_data['Cluster'] = km.fit_predict(self.scaled)
        print(f'\n✅ KMeans fitted with k={self.optimal_k}')
        print('\nCluster summary (mean values):')
        print(self.seg_data.groupby('Cluster').mean().round(2))

    def _name_clusters(self):
        # Auto-name based on churn probability rank
        churn_by_cluster = self.seg_data.groupby('Cluster')['Churn Probability'].mean()
        sorted_clusters = churn_by_cluster.sort_values().index.tolist()
        auto_names = {
            sorted_clusters[0]: 'Loyal Low-Risk Customers',
            sorted_clusters[-1]: 'High-Risk Customers'
        }
        if len(sorted_clusters) > 2:
            for c in sorted_clusters[1:-1]:
                auto_names[c] = 'Medium-Risk Customers'
        self.seg_data['Segment'] = self.seg_data['Cluster'].map(auto_names)
        print('\n🏷️  Auto-named segments based on Churn Probability:')
        for c, name in auto_names.items():
            avg = churn_by_cluster[c]
            print(f'   Cluster {c} → "{name}"  (avg churn prob = {avg:.2f})')

    def _visualise(self):
        fig, axes = plt.subplots(1, 3, figsize=(16, 5))
        pairs = [
            ('Monthly Charges', 'Churn Probability'),
            ('Tenure Months',   'Churn Probability'),
            ('Total Charges',   'Churn Probability')
        ]
        for ax, (xc, yc) in zip(axes, pairs):
            for seg, grp in self.seg_data.groupby('Segment'):
                ax.scatter(grp[xc], grp[yc], label=seg, alpha=0.5, s=10)
            ax.set_xlabel(xc)
            ax.set_ylabel(yc)
            ax.set_title(f'{xc} vs Churn Prob')
            ax.legend(fontsize=7)
        plt.suptitle('Customer Segments', fontsize=13)
        plt.tight_layout()
        plt.show()

        # Segment size
        seg_counts = self.seg_data['Segment'].value_counts()
        plt.figure(figsize=(6, 4))
        seg_counts.plot(kind='bar', color=['steelblue', 'salmon', 'green'])
        plt.title('Number of Customers per Segment')
        plt.ylabel('Count')
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()


# ▶ RUN AGENT 4
seg_agent = SegmentationAgent(raw_df, best_model, X)
seg_data = seg_agent.run()

---
## 💡 AGENT 5 — Business Recommendations Agent
*Automatically generates actionable retention strategies per segment.*

In [ ]:
class BusinessRecommendationAgent:
    """
    Agent 5: Reads segment profiles and auto-generates
    business recommendations and revenue-at-risk estimates.
    """

    def __init__(self, seg_data):
        self.seg_data = seg_data

    def run(self):
        print('=' * 60)
        print('💡 AGENT 5: BUSINESS RECOMMENDATIONS AGENT STARTED')
        print('=' * 60)

        self._segment_profiles()
        self._revenue_at_risk()
        self._generate_recommendations()
        self._priority_table()

        print('\n✅ AGENT 5 COMPLETE — Recommendations ready!')

    def _segment_profiles(self):
        self.profile = self.seg_data.groupby('Segment').agg(
            Customers       = ('Segment', 'count'),
            Avg_Churn_Prob  = ('Churn Probability', 'mean'),
            Avg_Tenure      = ('Tenure Months', 'mean'),
            Avg_Monthly     = ('Monthly Charges', 'mean'),
            Avg_Total       = ('Total Charges', 'mean')
        ).round(2)
        print('\n📊 Segment Profiles:')
        print(self.profile)

        # Bar chart of avg churn prob per segment
        plt.figure(figsize=(7, 4))
        self.profile['Avg_Churn_Prob'].sort_values().plot(
            kind='barh', color=['green', 'orange', 'red'])
        plt.title('Average Churn Probability per Segment')
        plt.xlabel('Churn Probability')
        plt.tight_layout()
        plt.show()

    def _revenue_at_risk(self):
        self.seg_data['Revenue at Risk'] = (
            self.seg_data['Monthly Charges'] * self.seg_data['Churn Probability']
        )
        rev_risk = self.seg_data.groupby('Segment')['Revenue at Risk'].sum().round(2)
        print('\n💰 Monthly Revenue at Risk per Segment:')
        for seg, val in rev_risk.sort_values(ascending=False).items():
            print(f'   {seg}: ${val:,.2f}')

        plt.figure(figsize=(7, 4))
        rev_risk.sort_values().plot(kind='barh', color=['green', 'orange', 'red'])
        plt.title('Monthly Revenue at Risk per Segment ($)')
        plt.xlabel('Revenue at Risk ($)')
        plt.tight_layout()
        plt.show()
        self.rev_risk = rev_risk

    def _generate_recommendations(self):
        STRATEGIES = {
            'High-Risk Customers': [
                '🚨 PRIORITY 1 — Immediate Action Required',
                '  → Offer personalised discount or loyalty reward',
                '  → Proactive outreach via phone/email within 48 hours',
                '  → Offer contract upgrade (Month-to-month → 1 or 2 year)',
                '  → Assign dedicated account manager'
            ],
            'Medium-Risk Customers': [
                '⚠️  PRIORITY 2 — Monitor & Engage',
                '  → Send satisfaction survey and address pain points',
                '  → Offer add-on services (streaming, security)',
                '  → Loyalty points or referral programme',
                '  → Re-engage with promotional emails'
            ],
            'Loyal Low-Risk Customers': [
                '✅ PRIORITY 3 — Retain & Upsell',
                '  → Upsell premium plans (higher speed, more features)',
                '  → Referral incentives to bring new customers',
                '  → Reward loyalty with anniversary offers',
                '  → Minimal intervention needed — keep satisfaction high'
            ]
        }

        print('\n' + '=' * 60)
        print('📋 BUSINESS RECOMMENDATIONS PER SEGMENT')
        print('=' * 60)
        for seg, lines in STRATEGIES.items():
            if seg in self.profile.index:
                n = int(self.profile.loc[seg, 'Customers'])
                cp = self.profile.loc[seg, 'Avg_Churn_Prob']
                print(f'\n🔹 {seg} ({n} customers | avg churn prob = {cp:.0%})')
                for line in lines:
                    print(line)

    def _priority_table(self):
        print('\n' + '=' * 60)
        print('📌 EXECUTIVE SUMMARY TABLE')
        print('=' * 60)
        summary = self.profile.copy()
        summary['Revenue_at_Risk'] = self.rev_risk
        summary['Action'] = summary.index.map(lambda s:
            'IMMEDIATE' if 'High' in s else
            ('MONITOR'   if 'Medium' in s else 'UPSELL')
        )
        print(summary[['Customers', 'Avg_Churn_Prob',
                        'Avg_Monthly', 'Revenue_at_Risk', 'Action']].to_string())


# ▶ RUN AGENT 5
rec_agent = BusinessRecommendationAgent(seg_data)
rec_agent.run()

---
## 🚀 MASTER PIPELINE — Run Everything in One Shot

In [ ]:
def run_full_pipeline(filepath='Telco_customer_churn.xlsx'):
    print('\n' + '🚀' * 30)
    print('   CHURN PREDICTION AGENTIC PIPELINE — FULL RUN')
    print('🚀' * 30 + '\n')

    # Agent 1
    eda   = AutoEDAAgent(filepath)
    df, report = eda.run()

    # Agent 2
    prep  = AutoPreprocessingAgent(df, report)
    X, Y, clean = prep.run()

    # Agent 3
    model = AutoModelSelectionAgent(X, Y, report.get('imbalanced', True))
    best, Xtr, Xte, Ytr, Yte = model.run()

    # Agent 4
    seg   = SegmentationAgent(df, best, X)
    seg_data = seg.run()

    # Agent 5
    rec   = BusinessRecommendationAgent(seg_data)
    rec.run()

    print('\n' + '✅' * 30)
    print('   PIPELINE COMPLETE!')
    print('✅' * 30)

# Uncomment the line below to run the entire pipeline from scratch in one shot:
# run_full_pipeline('Telco_customer_churn.xlsx')